# 项目三：各平台 ROI 预算重新分配

## 业务背景

在天猫、京东、得物、抖音四个平台同时投放广告，总预算固定但各平台 ROI 差异明显。
需要回答三个核心问题：

1. **各平台真实效能如何？**（当前 ROI + 边际 ROI 递减趋势）
2. **每个平台承担什么角色？**（四象限分析：拉新场 vs 种草地 vs 引流渠道）
3. **预算应该怎么调？**（基于边际效益的优化分配方案）

**方法论框架：**
1. 投放效能指标 × 四象限定位
2. 多项式回归拟合投入-产出曲线，量化边际收益递减
3. 末次点击 + 时间衰减双模型归因分析
4. 综合 ROI + 边际 ROI 的预算优化

**数据源：** `ad_campaigns` + `attribution_data`

---
## 1. 环境准备与数据获取

## 数据表结构说明

本项目使用 **2 张表**，模拟一家百人规模电商在天猫、京东、得物、抖音四个平台的
全年广告投放数据。总月均广告费约 53 万元，年投放约 640 万元。

### 表关系总览

```
ad_campaigns (广告投放记录)          attribution_data (用户触点归因)
  └── campaign_id PK                 └── id PK
  └── platform (天猫/京东/得物/抖音)   └── user_id → 关联用户
  └── 每次投放的花费+产出             └── touchpoint + platform → 触点来源
                                     └── conversion_time → 是否最终转化
```

> 两张表**独立使用**：
> `ad_campaigns` 用于效能指标和边际 ROI 回归，
> `attribution_data` 用于跨平台归因分析。

---

### 1. `ad_campaigns` — 广告投放记录表

| 字段 | 类型 | 含义 |
|------|------|------|
| `campaign_id` | INT PK | 投放活动唯一标识 |
| `platform` | TEXT | 投放平台（天猫/京东/得物/抖音） |
| `campaign_type` | TEXT | 投放类型（5 种） |
| `campaign_date` | TEXT (日期) | 投放日期 |
| `spend` | REAL | 投放花费（元） |
| `impressions` | INT | 曝光次数 |
| `clicks` | INT | 点击次数 |
| `conversions` | INT | 转化次数（下单） |
| `revenue` | REAL | 转化带来的收入（元） |

**`campaign_type` 五种投放类型：**

| 类型 | 含义 | 典型场景 |
|------|------|---------|
| 搜索广告 | 关键词竞价排名 | 用户主动搜索时触达 |
| 信息流 | 原生内容推荐 | 被动触达，拉新为主 |
| 达人带货 | KOL/KOC 合作推广 | 信任背书，高转化 |
| 品牌专区 | 品牌搜索结果页 | 品牌词防守 |
| 直播推广 | 直播间投流 | 冲动消费，高 GMV |

**各平台参数（数据生成基础）：**

| 平台 | 月均预算 | 基础 CPC | 基础 CVR | 基础 CTR | 产出系数 |
|------|---------|---------|---------|---------|---------|
| 天猫 | ¥180,000 | ¥2.8 | 3.5% | 3.5% | 3,200 |
| 京东 | ¥120,000 | ¥3.2 | 2.8% | 2.8% | 2,500 |
| 得物 | ¥80,000 | ¥1.6 | 4.5% | 4.0% | 3,600 |
| 抖音 | ¥150,000 | ¥1.3 | 1.8% | 2.5% | 2,200 |

**投入-产出模型：**
```
revenue = A × ln(spend + 1) + 噪声（正态分布, σ=25%）
```
使用对数函数模拟**边际收益递减**——花费越多，每多花 1 元的额外收入越少。

> **数据特点**：
> - 大促月份（6月、11月）预算翻倍，投放次数翻倍
> - 每个平台每月 15-30 条 campaign，全年约 800+ 条记录
> - 产出系数 A 经过校准，全域 ROI 约 1.5-3.5（符合真实电商水平）

---

### 2. `attribution_data` — 用户触点归因表

| 字段 | 类型 | 含义 |
|------|------|------|
| `id` | INT PK | 记录唯一标识 |
| `user_id` | INT | 用户标识 |
| `touchpoint` | TEXT | 触点平台（用户在哪看到广告） |
| `platform` | TEXT | 转化平台（用户在哪下单） |
| `event_time` | TEXT (datetime) | 触点发生时间 |
| `conversion_time` | TEXT (datetime) | 转化发生时间（NULL=未转化） |
| `days_to_conversion` | INT | 从触点到转化的天数（NULL=未转化） |

**跨平台转化路径（数据生成权重）：**

| 触点 → 转化 | 概率 | 平均转化天数 | 业务含义 |
|------------|------|------------|---------|
| 抖音 → 天猫 | 28% | 3 天 | 抖音种草，天猫成交（最常见） |
| 得物 → 天猫 | 22% | 5 天 | 得物看潮流，天猫比价下单 |
| 抖音 → 京东 | 18% | 4 天 | 抖音引流到京东 |
| 抖音 → 抖音 | 14% | 1 天 | 抖音闭环（直播带货） |
| 天猫 → 天猫 | 10% | 2 天 | 天猫站内转化 |
| 得物 → 京东 | 8% | 6 天 | 得物种草，京东成交 |
| 京东 → 京东 | 3% | 1 天 | 京东站内转化 |
| 得物 → 得物 | 2% | 1 天 | 得物闭环 |
| 天猫 → 京东 | 5% | 3 天 | 天猫浏览，京东下单 |

> **为什么需要归因分析？**
>
> 如果只用"末次点击归因"，一个用户先在抖音看广告、再到天猫搜索、
> 最后在京东下单——转化功劳全算京东的。但显然抖音和天猫也参与了引导。
> 时间衰减归因能更公平地分配贡献：离转化越近的触点权重越大（半衰期 3 天）。

---

### 两张表的分工

| 分析模块 | 使用的表 | 原因 |
|---------|---------|------|
| 平台效能指标（ROI/CPA/CTR/CVR） | `ad_campaigns` | 花费和产出数据都在这里 |
| 四象限角色定位 | `ad_campaigns` | 用 CPC 和 ROI 定位 |
| 边际 ROI 回归 | `ad_campaigns` | 按 platform 分组做多项式回归 |
| 跨平台归因 | `attribution_data` | 追踪用户多触点路径 |
| 预算优化 | `ad_campaigns`（ROI + 边际 ROI） | 综合两个维度分配预算 |

In [ ]:
# ══════════════════════════════════════════════════════════
# 环境准备 & 数据库连接（支持 SQLite 和 MySQL 双模式）
# ══════════════════════════════════════════════════════════
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from datetime import datetime

sns.set_theme(style='whitegrid', context='talk', font='Microsoft YaHei')
plt.rcParams['axes.unicode_minus'] = False

# 数据库选择: 改为 'mysql' 即可切换到 MySQL
DB_TYPE = 'sqlite'  # 可选: 'sqlite' 或 'mysql'

if DB_TYPE == 'sqlite':
    conn = sqlite3.connect('roi_allocation.db')
    print('[SQLite] roi_allocation.db')
elif DB_TYPE == 'mysql':
    import pymysql
    conn = pymysql.connect(
        host='localhost', user='root', password='123456',
        database='ecommerce_analysis', charset='utf8mb4',
    )
    print('[MySQL] ecommerce_analysis')

In [ ]:
# 广告投放数据
campaigns_df = pd.read_sql('SELECT * FROM ad_campaigns', conn)
campaigns_df['campaign_date'] = pd.to_datetime(campaigns_df['campaign_date'])

# 归因数据
attr_df = pd.read_sql('SELECT * FROM attribution_data', conn)
attr_df['event_time'] = pd.to_datetime(attr_df['event_time'])
attr_df['conversion_time'] = pd.to_datetime(attr_df['conversion_time'])

print(f'广告投放: {len(campaigns_df)} 条 campaign，{campaigns_df["platform"].nunique()} 个平台')
print(f'归因数据: {len(attr_df)} 条触点记录')
print(f'\n各平台 campaign 数量:')
print(campaigns_df['platform'].value_counts())

---
## 2. 各平台效能指标汇总

计算六大核心指标，形成各平台的完整效能画像：

| 指标 | 公式 | 含义 |
|------|------|------|
| CTR | Clicks / Impressions | 广告吸引力 |
| CVR | Conversions / Clicks | 落地页转化力 |
| CPA | Spend / Conversions | 获客成本 |
| CPC | Spend / Clicks | 点击成本 |
| ROI | Revenue / Spend | 投资回报率 |
| ARPU | Revenue / Conversions | 单用户价值 |

In [ ]:
# ============================================================
# 各平台效能指标汇总 — 从 campaign 明细到平台级 KPI
# ============================================================
# 六大指标的关系:
#   CPC(点击成本) = Spend / Clicks           ← 每次点击花多少钱
#   CTR(点击率)   = Clicks / Impressions     ← 曝光中有多少人点击
#   CVR(转化率)   = Conversions / Clicks     ← 点击的人中有多少下单
#   CPA(获客成本) = Spend / Conversions      ← 获取一个下单用户花多少钱
#   ROI(投资回报) = Revenue / Spend          ← 每花 1 元挣回多少
#   ARPU(用户价值)= Revenue / Conversions    ← 每个下单用户平均贡献多少收入
#
# 关系: CPA = CPC / CVR  (点击成本 ÷ 落地页转化率)

platform_metrics = (
    campaigns_df.groupby('platform')
    .agg(
        # 按平台汇总: sum() 把所有 campaign 的数据加总
        total_spend=('spend', 'sum'),
        total_impressions=('impressions', 'sum'),
        total_clicks=('clicks', 'sum'),
        total_conversions=('conversions', 'sum'),
        total_revenue=('revenue', 'sum'),
    )
    .reset_index()
)

# 计算派生指标
platform_metrics['CTR']   = platform_metrics['total_clicks'] / platform_metrics['total_impressions']
platform_metrics['CVR']   = platform_metrics['total_conversions'] / platform_metrics['total_clicks']
platform_metrics['CPA']   = platform_metrics['total_spend'] / platform_metrics['total_conversions']
platform_metrics['CPC']   = platform_metrics['total_spend'] / platform_metrics['total_clicks']
platform_metrics['ROI']   = platform_metrics['total_revenue'] / platform_metrics['total_spend']
# ARPU: 每个转化用户平均带来的收入，衡量用户质量
platform_metrics['ARPU']  = platform_metrics['total_revenue'] / platform_metrics['total_conversions']

print('各平台核心效能指标:')
# >6s 右对齐占6格, >10s 右对齐占10格
print(f'{"平台":>6s}  {"花费":>10s}  {"收入":>10s}  {"ROI":>6s}  {"CTR":>6s}  {"CVR":>6s}  {"CPA":>6s}  {"CPC":>6s}')
print('-' * 80)
for _, row in platform_metrics.iterrows():
    print(f'{row["platform"]:>6s}  {row["total_spend"]:>10,.0f}  {row["total_revenue"]:>10,.0f}  '
          f'{row["ROI"]:>5.2f}  {row["CTR"]:>5.1%}  {row["CVR"]:>5.1%}  '
          f'{row["CPA"]:>6,.0f}  {row["CPC"]:>6.2f}')

---
## 3. 四象限分析：平台角色定位

以 **CPC（点击成本）** 为横轴，**ROI** 为纵轴，
将四个平台划分到四个象限中，每个象限对应不同的战略角色：

- 左上（低成本+高回报）→ **高效拉新场**
- 右上（高成本+高回报）→ **高价值种草地**
- 左下（低成本+低回报）→ **低成本引流**
- 右下（高成本+低回报）→ **需要优化或削减**

In [ ]:
cpc_mean = platform_metrics['CPC'].mean()
roi_mean = platform_metrics['ROI'].mean()

print(f'CPC 均值: {cpc_mean:.2f}   ROI 均值: {roi_mean:.2f}')
print(f'\n各平台角色定位:')
for _, row in platform_metrics.iterrows():
    if row['CPC'] < cpc_mean and row['ROI'] > roi_mean:
        role = '高效拉新场'
    elif row['CPC'] < cpc_mean:
        role = '低成本引流'
    elif row['ROI'] > roi_mean:
        role = '高价值种草地'
    else:
        role = '需优化'
    print(f'  {row["platform"]:>6s}: CPC={row["CPC"]:.2f}  ROI={row["ROI"]:.2f}  →  {role}')

---
## 4. 边际 ROI 回归分析

### 为什么不能只看平均 ROI？

平均 ROI = 总收入 / 总花费，是一个「回顾性」指标。
但投放存在**边际收益递减**——每多花 1 元带来的额外收入会越来越少。

**方法：** 使用二次多项式回归 `revenue = a·spend² + b·spend + c` 拟合投入-产出曲线，
则边际收入 = 2a·spend + b，当前边际 ROI = 2a·avg_spend + b。

当边际 ROI < 1 时，继续追加投放将亏损——此时应削减该渠道预算。

In [ ]:
# ============================================================
# 边际 ROI 回归分析 — 量化"每多投 1 元能多挣多少"
# ============================================================
# 为什么这很重要?
#   平均 ROI = 总收入/总花费 → 这是"回头看"，告诉你过去整体效率
#   边际 ROI = 最后一元钱带来的收入 → 这是"往前看"，告诉你该不该继续投
#
# 经济学原理: 边际收益递减
#   第一个 1 万元广告 → 带来 5 万收入 (ROI=5)
#   第二个 1 万元     → 带来 3 万收入 (边际ROI=3)
#   第十个 1 万元     → 可能只带来 0.8 万收入 (边际ROI<1, 该停了)
#
# 数学方法: 二次多项式 regression
#   revenue = a × spend² + b × spend + c
#   边际收入 = 导数 = 2a × spend + b
#   边际 ROI = 边际收入(因为每多投 1 元)

def calculate_marginal_roi_regression(df):
    """使用二次多项式回归拟合各平台的投入-产出曲线"""
    results = {}
    for platform in df['platform'].unique():
        pdata = df[df['platform'] == platform].sort_values('spend')
        if len(pdata) < 4:   # 样本太少，跳过
            continue

        X = pdata[['spend']].values   # 自变量: 花费
        y = pdata['revenue'].values   # 因变量: 收入

        # PolynomialFeatures(degree=2): 把 [spend] 变成 [1, spend, spend²]
        # 这样 LinearRegression 就能拟合二次函数了
        poly = PolynomialFeatures(degree=2, include_bias=True)
        X_poly = poly.fit_transform(X)

        model = LinearRegression()
        model.fit(X_poly, y)           # 拟合: revenue = a*spend² + b*spend + c

        # model.coef_[0]=常数项系数, [1]=spend系数(b), [2]=spend²系数(a)
        a, b, c = model.coef_[2], model.coef_[1], model.intercept_

        avg_spend = X.mean()           # 当前平均花费水平
        marginal_revenue = 2 * a * avg_spend + b  # 在当前水平下的边际收入
        marginal_roi = max(0, marginal_revenue)    # 边际 ROI（截断到 >=0）

        # R²: 模型拟合优度，0~1，越接近 1 拟合越好
        y_pred = model.predict(X_poly)
        ss_res = np.sum((y - y_pred) ** 2)   # 残差平方和
        ss_tot = np.sum((y - y.mean()) ** 2) # 总平方和
        r2 = 1 - ss_res / max(ss_tot, 1e-10)

        results[platform] = {
            'a': a, 'b': b, 'c': c,
            'marginal_roi': marginal_roi,  # 当前边际 ROI
            'r2': r2,                       # 模型可信度
            'model': model, 'poly': poly,
            'avg_spend': avg_spend,
            'spend_range': (X.min(), X.max()),
        }
    return results


marginal_results = calculate_marginal_roi_regression(campaigns_df)

print('各平台边际 ROI 分析:')
print(f'{"平台":>6s}  {"平均ROI":>6s}  {"边际ROI":>6s}  {"R²":>6s}  {"投入区间":>20s}')
print('-' * 60)
for plat, res in marginal_results.items():
    avg_roi = platform_metrics[platform_metrics['platform'] == plat]['ROI'].values[0]
    print(f'{plat:>6s}  {avg_roi:>5.2f}  {res["marginal_roi"]:>5.2f}  {res["r2"]:>5.3f}  '
          f'{res["spend_range"][0]:>8,.0f}-{res["spend_range"][1]:>8,.0f}')

---
## 5. 跨平台归因分析

一个用户可能在抖音看到广告，在天猫搜索，在京东下单——传统"末次点击归因"
会把这笔转化全归给京东，但显然不公平。

我们实现两种归因模型：
1. **末次点击归因**：转化功劳全归最后一个触点（基准模型）
2. **时间衰减归因**：越接近转化时间的触点获得越大权重
   - 权重 = 2^(-days_to_conversion / 3)，半衰期 3 天

In [ ]:
def attribution_analysis(attr_df):
    """末次点击 + 时间衰减双模型归因"""
    converted = attr_df.dropna(subset=['conversion_time']).copy()
    if len(converted) == 0:
        return pd.DataFrame(), pd.DataFrame()
    
    # 末次点击归因
    last_click = (
        converted.sort_values('conversion_time')
        .groupby('user_id')
        .last()
        .reset_index()
    )
    last_click_attr = (
        last_click.groupby('platform')
        .agg(last_click_conversions=('user_id', 'count'))
        .reset_index()
    )
    
    # 时间衰减归因
    halflife = 3
    converted['decay_weight'] = 2.0 ** (-converted['days_to_conversion'].fillna(30) / halflife)
    user_weights = converted.groupby('user_id')['decay_weight'].sum().reset_index(name='total_weight')
    converted = converted.merge(user_weights, on='user_id')
    converted['normalized_weight'] = converted['decay_weight'] / converted['total_weight']
    
    time_decay_attr = (
        converted.groupby('platform')
        .agg(time_decay_conversions=('normalized_weight', 'sum'))
        .reset_index()
    )
    
    attribution = last_click_attr.merge(time_decay_attr, on='platform', how='outer').fillna(0)
    return attribution, converted


attribution, converted = attribution_analysis(attr_df)

if len(attribution) > 0:
    print('跨平台归因贡献对比:')
    print(f'{"平台":>6s}  {"末次点击":>10s}  {"时间衰减":>10s}')
    print('-' * 35)
    for _, row in attribution.iterrows():
        print(f'{row["platform"]:>6s}  {row["last_click_conversions"]:>10,.0f}  '
              f'{row["time_decay_conversions"]:>10.1f}')

---
## 6. 预算优化方案

### 优化策略

综合两个维度决定各平台预算占比：
- **平均 ROI**（权重 60%）：反映历史整体效率
- **边际 ROI**（权重 40%）：反映当前追加投放的边际收益

边际 ROI 权重较低的原因是：模型拟合存在不确定性（看 R²），
且边际 ROI 对数据质量更敏感。

In [ ]:
def optimize_budget(platform_metrics, marginal_results, total_budget):
    """综合 ROI + 边际 ROI 计算建议预算分配"""
    platforms = platform_metrics['platform'].tolist()
    current = platform_metrics.set_index('platform')['total_spend'].to_dict()
    total_current = sum(current.values())
    
    roi_values = platform_metrics.set_index('platform')['ROI']
    marginal_values = pd.Series({
        p: marginal_results.get(p, {}).get('marginal_roi', 1.0)
        for p in platforms
    })
    
    roi_norm = roi_values / roi_values.sum()
    marginal_norm = marginal_values / marginal_values.sum()
    combined_score = 0.6 * roi_norm + 0.4 * marginal_norm
    suggested_share = combined_score / combined_score.sum()
    
    return pd.DataFrame({
        'platform': platforms,
        'current_spend': [current[p] for p in platforms],
        'current_share': [current[p] / total_current for p in platforms],
        'suggested_share': suggested_share.values,
        'suggested_spend': [total_budget * s for s in suggested_share.values],
        'roi': roi_values.values,
        'marginal_roi': marginal_values.values,
    })


total_budget = platform_metrics['total_spend'].sum()
optimization = optimize_budget(platform_metrics, marginal_results, total_budget)

print(f'总预算: {total_budget:,.0f} 元\n')
print('预算分配建议:')
print(f'{"平台":>6s}  {"当前占比":>8s}  {"建议占比":>8s}  {"调整":>8s}  {"ROI":>6s}  {"边际ROI":>6s}')
print('-' * 60)
for _, row in optimization.iterrows():
    change = (row['suggested_share'] - row['current_share']) * 100
    direction = '+' if change > 0 else ''
    print(f'{row["platform"]:>6s}  {row["current_share"]:>7.0%}  {row["suggested_share"]:>7.0%}  '
          f'{direction}{change:>+5.1f}pp  {row["roi"]:>5.2f}  {row["marginal_roi"]:>5.2f}')

# 预估优化后全域 ROI
current_revenue = platform_metrics['total_revenue'].sum()
suggested_revenue = sum(row['suggested_spend'] * row['roi'] for _, row in optimization.iterrows())
# 考虑边际递减：实际提升打 8 折
realistic_revenue = suggested_revenue * 0.8 + current_revenue * 0.2
new_roi = realistic_revenue / total_budget
current_roi = current_revenue / total_budget
print(f'\n全域 ROI 预估: {current_roi:.2f} → {new_roi:.2f} (提升 {(new_roi/current_roi-1):.1%})')

---
## 7. 综合可视化看板

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(22, 14))
colors_bar = sns.color_palette('Blues_r', len(platform_metrics))

# 图 1: 各平台 ROI 对比
ax = axes[0, 0]
bars = ax.bar(platform_metrics['platform'], platform_metrics['ROI'], color=colors_bar, edgecolor='white')
ax.axhline(1.0, color='red', linestyle='--', alpha=0.5, label='盈亏线 (ROI=1)')
ax.set_title('各平台 ROI 对比', fontweight='bold'); ax.set_ylabel('ROI'); ax.legend()
for bar, roi in zip(bars, platform_metrics['ROI']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05, f'{roi:.2f}', ha='center', fontweight='bold')

# 图 2: 四象限分析
ax = axes[0, 1]
for _, row in platform_metrics.iterrows():
    ax.scatter(row['CPC'], row['ROI'], s=row['total_spend'] / 5000, alpha=0.7)
    ax.annotate(row['platform'], (row['CPC'], row['ROI']), fontsize=12, fontweight='bold',
                xytext=(8, 8), textcoords='offset points')
ax.axhline(roi_mean, color='gray', linestyle='--', alpha=0.5)
ax.axvline(cpc_mean, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('CPC (点击成本)'); ax.set_ylabel('ROI')
ax.set_title('四象限分析: CPC vs ROI', fontweight='bold')

# 标注象限含义
ax.text(cpc_mean * 0.6, roi_mean * 1.1, '低成本高回报\n(高效拉新场)', ha='center', fontsize=9, color='green')
ax.text(cpc_mean * 1.4, roi_mean * 1.1, '高成本高回报\n(高价值种草地)', ha='center', fontsize=9, color='blue')
ax.text(cpc_mean * 0.6, roi_mean * 0.85, '低成本低回报\n(低成本引流)', ha='center', fontsize=9, color='orange')
ax.text(cpc_mean * 1.4, roi_mean * 0.85, '高成本低回报\n(需优化)', ha='center', fontsize=9, color='red')

# 图 3: 投入-产出曲线（含边际递减）
ax = axes[0, 2]
x_plot = np.linspace(0, 80000, 100)
colors_line = {'天猫': '#E91E63', '京东': '#2196F3', '得物': '#4CAF50', '抖音': '#FF9800'}
for plat, res in marginal_results.items():
    poly = res['poly']; model = res['model']
    X_plot_poly = poly.transform(x_plot.reshape(-1, 1))
    y_plot = np.maximum(model.predict(X_plot_poly), 0)
    ax.plot(x_plot, y_plot, color=colors_line.get(plat, 'gray'), linewidth=2,
            label=f"{plat} (R²={res['r2']:.2f})")
    ax.scatter([res['avg_spend']], [model.predict(poly.transform([[res['avg_spend']]]))[0]],
               color=colors_line.get(plat, 'gray'), s=100, zorder=5)
ax.set_xlabel('花费'); ax.set_ylabel('预估收入')
ax.set_title('投入-产出曲线（含边际递减）', fontweight='bold'); ax.legend(fontsize=9)

# 图 4: 预算分配对比
ax = axes[1, 0]
x = np.arange(len(platform_metrics)); width = 0.35
ax.bar(x - width / 2, optimization['current_share'], width, label='当前', color='#90CAF9')
ax.bar(x + width / 2, optimization['suggested_share'], width, label='建议', color='#1565C0')
ax.set_xticks(x); ax.set_xticklabels(optimization['platform'])
ax.set_ylabel('预算占比'); ax.set_title('预算分配对比', fontweight='bold')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0)); ax.legend()

# 图 5: 归因分析
ax = axes[1, 1]
if len(attribution) > 0:
    attr_plot = attribution.set_index('platform')
    attr_plot[['last_click_conversions', 'time_decay_conversions']].plot(
        kind='bar', ax=ax, color=['#42A5F5', '#FF7043'], edgecolor='white')
    ax.set_title('跨平台归因：末次点击 vs 时间衰减', fontweight='bold')
    ax.set_ylabel('转化贡献')
    ax.legend(['末次点击归因', '时间衰减归因'], fontsize=9)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=0)

# 图 6: CPA 对比
ax = axes[1, 2]
bars = ax.bar(platform_metrics['platform'], platform_metrics['CPA'], color=colors_bar, edgecolor='white')
ax.set_title('各平台 CPA 对比', fontweight='bold'); ax.set_ylabel('CPA (元)')
for bar, cpa in zip(bars, platform_metrics['CPA']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5, f'{cpa:.0f}', ha='center', fontweight='bold')

fig.suptitle('各平台 ROI 预算重新分配分析看板', fontsize=18, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('各平台ROI预算重新分配分析.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. 实施计划与关键结论

### 三阶段推进

| 阶段 | 时间 | 动作 | 风险控制 |
|------|------|------|----------|
| 第一阶段 | 第1-2周 | 将预算从低效渠道转移 10% 到高效渠道 | 建立实时监控看板，按日追踪指标 |
| 第二阶段 | 第3-4周 | 根据测试结果调整转移比例，优化各平台内投放模式 | A/B 验证 |
| 第三阶段 | 第2个月起 | 月度预算复盘，引入季节性因子 | 闭环监控机制 |

### 预期效果
- 全域营销 ROI 提升 15-25%
- 整体获客成本(CPA)降低 15% 以上
- 高效渠道预算占比提升 15%

In [ ]:
conn.close()
print('分析完成！')